# ── Cell 1: Bootstrap repo root + SPEC_HASH assertion ────────────────────────
import hashlib, json, os, sys, time
from pathlib import Path

root = Path.cwd()
for _ in range(10):
    if (root / 'research' / '__init__.py').exists():
        break
    root = root.parent
else:
    raise RuntimeError('repo root not found — launch Jupyter from inside TRADING-BOT/')

os.chdir(root)
sys.path.insert(0, str(root))

from research.experiments.exp10_proxy import CONFIG
assert hashlib.sha256(
    json.dumps(CONFIG, sort_keys=True).encode()
).hexdigest()[:12] == '7a809aaca7f3', 'CONFIG no coincide con spec v1 FROZEN'

print(f'repo root : {root}')
print('SPEC_HASH : 7a809aaca7f3  OK')

In [2]:
# ── Cell 2: Imports + constants ──────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from research.experiments.exp10_proxy import (
    compute_bin_stats,
    detect_events,
    match_controls,
    compute_directed_returns,
    paired_permutation_test,
    paired_bootstrap_ci,
    classify_outcome,
)

TAPE_PATH     = root / 'data' / 'raw' / 'BTCUSDT_AGGTRADES.csv'
HORIZON_MS    = CONFIG['horizon_primary_ms']   # 300_000 ms = 5 min
N_PERM        = 10_000
N_BOOT        = CONFIG['bootstrap_n_resamples']   # 2_000
RNG_SEED      = 42
MIN_N_SEGMENT = 15   # F4: n ≥ 15/bucket required

print(f'tape      : {TAPE_PATH}')
print(f'horizon   : {HORIZON_MS / 60_000:.0f} min')
print(f'N_PERM={N_PERM:,}   N_BOOT={N_BOOT:,}   MIN_N={MIN_N_SEGMENT}')

tape      : C:\Users\Lenovo\Documents\TRADING-BOT\data\raw\BTCUSDT_AGGTRADES.csv
horizon   : 5 min
N_PERM=10,000   N_BOOT=2,000   MIN_N=15


In [3]:
# ── Cell 3: Pipeline M1→M4 (∼2–10 min, dominado por I/O del tape) ───────────
t0 = time.time()

tape = pd.read_csv(
    TAPE_PATH,
    usecols=['timestamp_ms', 'price', 'qty', 'side'],
    dtype={'timestamp_ms': np.int64, 'price': np.float64,
           'qty': np.float64, 'side': str},
)
print(f'tape loaded     : {len(tape):>12,} trades  ({time.time()-t0:.1f}s)')

bins_df          = compute_bin_stats(tape)
events_df        = detect_events(bins_df)
matched_pairs_df = match_controls(events_df, bins_df)
returns_df, _    = compute_directed_returns(
    matched_pairs_df,
    tape[['timestamp_ms', 'price']],
    horizon_ms=HORIZON_MS,
)

print(f'N_bins          : {len(bins_df):>8,}')
print(f'N_events        : {len(events_df):>8,}')
print(f'N_pairs_matched : {len(matched_pairs_df):>8,}')
print(f'N_pairs_valid   : {len(returns_df):>8,}')
print(f'total time      : {time.time()-t0:.1f}s')

tape loaded     :   41,544,041 trades  (15.7s)
N_bins          :   86,400
N_events        :       78
N_pairs_matched :       78
N_pairs_valid   :       78
total time      : 55.5s


In [4]:
# ── Cell 4: Segmentation helpers ───────────────────────────────────────────
def run_segment(df_seg: pd.DataFrame, label: str) -> dict:
    n = len(df_seg)
    if n < MIN_N_SEGMENT:
        return {'seg': label, 'N': n, 'delta': None,
                'p': None, 'ci_lo': None, 'ci_hi': None,
                'outcome': f'N/A (N={n}<{MIN_N_SEGMENT})'}
    delta, p     = paired_permutation_test(df_seg, n_permutations=N_PERM, seed=RNG_SEED)
    ci_lo, ci_hi = paired_bootstrap_ci(df_seg, n_resamples=N_BOOT, seed=RNG_SEED)
    return {
        'seg': label, 'N': n,
        'delta': delta, 'p': p,
        'ci_lo': ci_lo, 'ci_hi': ci_hi,
        'outcome': classify_outcome(delta, p, ci_lo, ci_hi),
    }

def _session_label(t_bin: int) -> str:
    hour = (int(t_bin) % 86_400_000) // 3_600_000
    if hour < 8:  return 'Asia'
    if hour < 16: return 'London'
    return 'NY'

def _print_seg_table(rows: list, title: str) -> None:
    hdr = f'  {"Segmento":<10}  {"N":>4}  {"\u0394_obs":>10}  {"p_perm":>7}  {"IC_95 [lo, hi]":>30}  outcome'
    sep = f'  {"-"*10}  {"-"*4}  {"-"*10}  {"-"*7}  {"-"*30}  {"-"*6}'
    print(title)
    print(hdr)
    print(sep)
    for r in rows:
        if r['delta'] is None:
            print(f'  {r["seg"]:<10}  {r["N"]:>4}  {"":>10}  {"":>7}  {"":>30}  {r["outcome"]}')
        else:
            ci = f'[{r["ci_lo"]:+.5f}, {r["ci_hi"]:+.5f}]'
            print(f'  {r["seg"]:<10}  {r["N"]:>4}  {r["delta"]:>+10.6f}  {r["p"]:>7.4f}  {ci:>30}  {r["outcome"]}')

print('helpers ready')

helpers ready


In [5]:
# ── Cell 5: S1 — Direction (BUY vs SELL) ─────────────────────────────────
rows_s1 = []
for sign, label in [(1, 'BUY'), (-1, 'SELL')]:
    seg = returns_df[returns_df['event_sign_W'] == sign]
    rows_s1.append(run_segment(seg, label))

_print_seg_table(rows_s1, 'S1 — Direction (BUY vs SELL)')
print(f'  note: BUY={rows_s1[0]["N"]} events, SELL={rows_s1[1]["N"]} events')

S1 — Direction (BUY vs SELL)
  Segmento       N       Δ_obs   p_perm                  IC_95 [lo, hi]  outcome
  ----------  ----  ----------  -------  ------------------------------  ------
  BUY           40   -0.000481   0.9053            [-0.00115, +0.00016]  NULO
  SELL          38   +0.000311   0.3089            [-0.00080, +0.00145]  NULO
  note: BUY=40 events, SELL=38 events


In [6]:
# ── Cell 6: S3 — Session UTC (Asia / London / NY) ────────────────────────
returns_df = returns_df.copy()
returns_df['session'] = returns_df['event_t_bin'].apply(_session_label)

session_counts = returns_df['session'].value_counts().to_dict()
print(f'session distribution: {session_counts}')
print()

rows_s3 = []
for sess in ['Asia', 'London', 'NY']:
    seg = returns_df[returns_df['session'] == sess]
    rows_s3.append(run_segment(seg, sess))

_print_seg_table(rows_s3, 'S3 — Session UTC')

session distribution: {'London': 30, 'Asia': 27, 'NY': 21}

S3 — Session UTC
  Segmento       N       Δ_obs   p_perm                  IC_95 [lo, hi]  outcome
  ----------  ----  ----------  -------  ------------------------------  ------
  Asia          27   +0.000141   0.4078            [-0.00091, +0.00127]  NULO
  London        30   -0.000207   0.6226            [-0.00144, +0.00109]  NULO
  NY            21   -0.000239   0.6930            [-0.00109, +0.00060]  NULO


In [7]:
# ── Cell 7: S4 — Local intensity quartiles ───────────────────────────────
# Quartiles of |net_quote_W| among the N=78 events.
# Q1 = least extreme, Q4 = most extreme bursts.
# F4 guardrail: n ≥ 15/bucket required for WEAK claim.
returns_df['s4_quartile'] = pd.qcut(
    returns_df['event_net_quote_W'].abs(),
    q=4, labels=[1, 2, 3, 4]
).astype(int)

q_counts = returns_df['s4_quartile'].value_counts().sort_index().to_dict()
print(f'quartile N distribution: {q_counts}')
print()

rows_s4 = []
for q in [1, 2, 3, 4]:
    seg = returns_df[returns_df['s4_quartile'] == q]
    rows_s4.append(run_segment(seg, f'Q{q}'))

_print_seg_table(rows_s4, 'S4 — Intensity quartiles (Q1=least extreme, Q4=most extreme)')

# Delta trend across quartiles
deltas = [r['delta'] for r in rows_s4 if r['delta'] is not None]
if len(deltas) == 4:
    trend = 'increasing' if deltas[-1] > deltas[0] else 'flat/decreasing'
    print(f'\n  Q1→Q4 delta trend: {[f"{d:+.6f}" for d in deltas]}  ({trend})')

quartile N distribution: {1: 20, 2: 19, 3: 19, 4: 20}

S4 — Intensity quartiles (Q1=least extreme, Q4=most extreme)
  Segmento       N       Δ_obs   p_perm                  IC_95 [lo, hi]  outcome
  ----------  ----  ----------  -------  ------------------------------  ------
  Q1            20   +0.000420   0.3050            [-0.00064, +0.00183]  NULO
  Q2            19   +0.000019   0.5020            [-0.00112, +0.00092]  NULO
  Q3            19   -0.000277   0.6281            [-0.00176, +0.00137]  NULO
  Q4            20   -0.000546   0.7850            [-0.00183, +0.00073]  NULO

  Q1→Q4 delta trend: ['+0.000420', '+0.000019', '-0.000277', '-0.000546']  (flat/decreasing)


In [8]:
# ── Cell 8: Summary + decision rule ───────────────────────────────────────
all_rows = (
    [dict(seg=r['seg'], segmentacion='S1_direction', **{k: r[k] for k in ('N','delta','p','ci_lo','ci_hi','outcome')}) for r in rows_s1] +
    [dict(seg=r['seg'], segmentacion='S3_session',   **{k: r[k] for k in ('N','delta','p','ci_lo','ci_hi','outcome')}) for r in rows_s3] +
    [dict(seg=r['seg'], segmentacion='S4_intensity', **{k: r[k] for k in ('N','delta','p','ci_lo','ci_hi','outcome')}) for r in rows_s4]
)

print('=' * 62)
print('EXP10 M6 SEGMENTACIONES  (SPEC_HASH=7a809aaca7f3)')
print('=' * 62)
print(f'  {"Seg":>15}  {"N":>4}  {"\u0394_obs":>10}  {"p_perm":>7}  outcome')
print(f'  {"-"*15}  {"-"*4}  {"-"*10}  {"-"*7}  {"-"*6}')
for r in all_rows:
    delta_str = f'{r["delta"]:>+10.6f}' if r['delta'] is not None else f'{"":>10}'
    p_str     = f'{r["p"]:>7.4f}'       if r['p']     is not None else f'{"":>7}'
    label = f'{r["segmentacion"].split("_")[0]}:{r["seg"]}'
    print(f'  {label:>15}  {r["N"]:>4}  {delta_str}  {p_str}  {r["outcome"]}')

# Decision rule
valid_rows = [r for r in all_rows if r['delta'] is not None]
any_fuerte = any(r['outcome'] == 'FUERTE' for r in valid_rows)
any_debil  = any(r['outcome'] == 'D\u00c9BIL'  for r in valid_rows)

print()
print('  DECISION RULE:')
if any_fuerte:
    conclusion = 'Caso C — señal fuerte. Replicación obligatoria antes de promover.'
elif any_debil:
    conclusion = 'Caso B — segmentación consistente pero subpotente. Candidato Exp10_proxy_v2.'
else:
    conclusion = 'Caso A — todos NULOS. Cerrar definitivamente la familia Exp10.'
print(f'  {conclusion}')
print('=' * 62)

EXP10 M6 SEGMENTACIONES  (SPEC_HASH=7a809aaca7f3)
              Seg     N       Δ_obs   p_perm  outcome
  ---------------  ----  ----------  -------  ------
           S1:BUY    40   -0.000481   0.9053  NULO
          S1:SELL    38   +0.000311   0.3089  NULO
          S3:Asia    27   +0.000141   0.4078  NULO
        S3:London    30   -0.000207   0.6226  NULO
            S3:NY    21   -0.000239   0.6930  NULO
            S4:Q1    20   +0.000420   0.3050  NULO
            S4:Q2    19   +0.000019   0.5020  NULO
            S4:Q3    19   -0.000277   0.6281  NULO
            S4:Q4    20   -0.000546   0.7850  NULO

  DECISION RULE:
  Caso A — todos NULOS. Cerrar definitivamente la familia Exp10.


In [9]:
# ── Cell 9: Export segment_results.csv / .json ────────────────────────────
import json as _json

out_dir = root / 'research' / 'experiments' / 'outputs'
out_dir.mkdir(parents=True, exist_ok=True)

records = [
    {
        'segmentacion': r['segmentacion'],
        'segment':      r['seg'],
        'N':            r['N'],
        'delta_obs':    r['delta'],
        'p_perm':       r['p'],
        'ci_low':       r['ci_lo'],
        'ci_high':      r['ci_hi'],
        'outcome':      r['outcome'],
    }
    for r in all_rows
]

df_out = pd.DataFrame(records)

csv_path  = out_dir / 'segment_results.csv'
json_path = out_dir / 'segment_results.json'

df_out.to_csv(csv_path, index=False, float_format='%.8f')

payload = {
    'spec_hash':    '7a809aaca7f3',
    'horizon_ms':   HORIZON_MS,
    'n_pairs_full': len(returns_df),
    'min_n_segment': MIN_N_SEGMENT,
    'conclusion':   conclusion,
    'segments':     records,
}
json_path.write_text(_json.dumps(payload, indent=2, default=str), encoding='utf-8')

print(f'exported → {csv_path.relative_to(root)}')
print(f'exported → {json_path.relative_to(root)}')
print(df_out.to_string(index=False))

exported → research\experiments\outputs\segment_results.csv
exported → research\experiments\outputs\segment_results.json
segmentacion segment  N  delta_obs  p_perm    ci_low  ci_high outcome
S1_direction     BUY 40  -0.000481  0.9053 -0.001149 0.000160    NULO
S1_direction    SELL 38   0.000311  0.3089 -0.000801 0.001454    NULO
  S3_session    Asia 27   0.000141  0.4078 -0.000913 0.001266    NULO
  S3_session  London 30  -0.000207  0.6226 -0.001443 0.001092    NULO
  S3_session      NY 21  -0.000239  0.6930 -0.001090 0.000604    NULO
S4_intensity      Q1 20   0.000420  0.3050 -0.000640 0.001826    NULO
S4_intensity      Q2 19   0.000019  0.5020 -0.001117 0.000919    NULO
S4_intensity      Q3 19  -0.000277  0.6281 -0.001757 0.001375    NULO
S4_intensity      Q4 20  -0.000546  0.7850 -0.001827 0.000733    NULO
